# Qwen-Image-2.1 Standalone Runner — Google Drive Edition (Zero-Download)

Run **Base Qwen-Image-2.1 (30–50 steps, CFG 3.5–5.0, negative prompts)** and **Viggle Turbo (4 steps)** on Google Colab by mounting your existing models directly from Google Drive.

### Key Features:
- **Zero Redownload**: Loads all GGUF weights, text encoders, vision projectors, and VAEs directly from your Google Drive folder.
- **Strict Validation**: Checks that all required model files exist on Drive and gives clear errors if anything is missing (never triggers automatic downloads).
- **Full Capabilities Preserved**: Text-to-Image Generation, Segmentation-Aware Pixel-Preserving Editing, and Global Image Editing.

In [ ]:
# 1. Verify NVIDIA GPU (Supports T4, L4, V100, A100)
!nvidia-smi

In [ ]:
# 2. Clone Repository, Install Dependencies & Build CUDA Engine
import os

if os.path.exists("/content"):
    if not os.path.exists("/content/vision-model"):
        !git clone https://github.com/samirpatil2000/vision-model.git /content/vision-model
    %cd /content/vision-model
    !git pull origin main
else:
    if not os.path.exists("vision-model"):
        !git clone https://github.com/samirpatil2000/vision-model.git
    %cd vision-model

# Install Python dependencies for segmentation and image processing
!pip install -q transformers scipy torch torchvision huggingface_hub pillow

# Compile stable-diffusion.cpp with CUDA acceleration
!chmod +x setup.sh download_models.sh run_inference.sh edit_image.sh edit_segmented.sh
!./setup.sh

# Verify compilation
if os.path.exists("./stable-diffusion.cpp/build/bin/sd-cli"):
    print("\n✓ SUCCESS: CUDA engine ('sd-cli') is built and ready!")
else:
    print("\n⚠️ sd-cli not found. Running direct build fallback...")
    !cd stable-diffusion.cpp && cmake --build build --config Release -j 2


In [ ]:
# @title 3. Mount Google Drive & Configure Storage Paths { run: "auto" }
import os
import sys

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"Note: Running outside Colab or Drive already mounted: {e}")

# 2. Configure path to your models folder in Google Drive
# Set this to the folder on your Drive containing the 'models' directory or model files:
drive_models_path = "/content/drive/MyDrive/vision-model/models" # @param {type:"string"}

# 3. Symlink local models folder to Google Drive for direct loading
local_models_dir = "/content/vision-model/models"
if os.path.exists("/content"):
    if os.path.islink(local_models_dir):
        os.unlink(local_models_dir)
    elif os.path.isdir(local_models_dir) and not os.listdir(local_models_dir):
        os.rmdir(local_models_dir)

    if os.path.exists(drive_models_path):
        if not os.path.exists(local_models_dir):
            os.symlink(drive_models_path, local_models_dir)
            print(f"✓ Linked {local_models_dir} -> {drive_models_path}")
        else:
            print(f"✓ Models directory active at: {local_models_dir}")
    else:
        print(f"⚠️ Folder not found on Drive: {drive_models_path}")
        print("Please verify your Drive path above and re-run this cell.")


In [ ]:
# @title 4. Verify Required Files on Google Drive (Strict Validation — No Downloads) { run: "auto" }
import os

# Select which model variant you plan to run
target_variant = "Base Model (Qwen-Image-2.1-Q4_K_M.gguf)" # @param ["Base Model (Qwen-Image-2.1-Q4_K_M.gguf)", "Turbo Model (Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf)"]

base_models_dir = "/content/vision-model/models"

if "Base" in target_variant:
    dit_name = "Qwen-Image-2.1-Q4_K_M.gguf"
else:
    dit_name = "Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf"

required_files = {
    "Diffusion Model (DiT)": os.path.join(base_models_dir, "diffusion_models", dit_name),
    "Text Encoder (Qwen3-VL)": os.path.join(base_models_dir, "text_encoders", "Qwen3VL-8B-Instruct-Q4_K_M.gguf"),
    "Vision Projector": os.path.join(base_models_dir, "text_encoders", "mmproj-Qwen3VL-8B-Instruct-F16.gguf"),
    "VAE (RGBA)": os.path.join(base_models_dir, "vae", "qwen_image_2.1_vae_bf16.safetensors"),
}

missing_files = []
print("=" * 65)
print(" Checking Required Model Files on Google Drive")
print("=" * 65)

for label, fpath in required_files.items():
    if os.path.exists(fpath):
        size_gb = os.path.getsize(fpath) / (1024 ** 3)
        print(f"✓ [FOUND]   {label:24} -> {os.path.basename(fpath)} ({size_gb:.2f} GB)")
    else:
        print(f"❌ [MISSING] {label:24} -> Expected at:\n   {fpath}")
        missing_files.append(fpath)

print("=" * 65)

if missing_files:
    error_msg = (
        f"\n❌ [ERROR] {len(missing_files)} required model file(s) are missing from Google Drive!\n"
        "Per configuration, automatic downloads are disabled.\n"
        "Please ensure the missing files exist in your Google Drive under:\n"
        f"   {drive_models_path}\n\n"
        "Required folder layout:\n"
        "   ├── diffusion_models/" + dit_name + "\n"
        "   ├── text_encoders/Qwen3VL-8B-Instruct-Q4_K_M.gguf\n"
        "   ├── text_encoders/mmproj-Qwen3VL-8B-Instruct-F16.gguf\n"
        "   └── vae/qwen_image_2.1_vae_bf16.safetensors\n"
    )
    raise FileNotFoundError(error_msg)
else:
    print("\n✓ All required files verified! You can run inference directly from Drive without downloading anything.")


In [ ]:
# @title 5. Text-to-Image Generation (Direct Drive Loading) { run: "auto" }
prompt = "A raw 35mm photograph of an elderly craftsman carving wood in a sunlit workshop in Florence, photorealistic" # @param {type:"string"}
negative_prompt = "blurry, low quality, distorted, extra limbs, cartoon" # @param {type:"string"}
model_type = "Base Model (Qwen-Image-2.1-Q4_K_M.gguf)" # @param ["Base Model (Qwen-Image-2.1-Q4_K_M.gguf)", "Turbo Model (Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf)"]
steps = 35 # @param {type:"slider", min:1, max:50, step:1}
cfg_scale = 4.5 # @param {type:"slider", min:1.0, max:12.0, step:0.5}
width = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
height = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
seed = 42 # @param {type:"integer"}
output_file = "./output/drive_generation.png" # @param {type:"string"}

import os
from IPython.display import Image, display

if not os.path.exists("./stable-diffusion.cpp/build/bin/sd-cli"):
    raise FileNotFoundError("Engine binary missing. Please run Cell 2 first.")

if "Base" in model_type:
    os.environ["MODEL"] = "./models/diffusion_models/Qwen-Image-2.1-Q4_K_M.gguf"
else:
    os.environ["MODEL"] = "./models/diffusion_models/Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf"

os.environ["STEPS"] = str(steps)
os.environ["CFG"] = str(cfg_scale)
os.environ["WIDTH"] = str(width)
os.environ["HEIGHT"] = str(height)
os.environ["SEED"] = str(seed)
if negative_prompt.strip():
    os.environ["NEGATIVE_PROMPT"] = negative_prompt.strip()
else:
    os.environ.pop("NEGATIVE_PROMPT", None)

!./run_inference.sh "{prompt}" "{output_file}"

if os.path.exists(output_file):
    display(Image(output_file))


In [ ]:
# @title 6. Segmentation-Aware Pixel-Preserving Edit (Direct Drive Loading) { run: "auto" }
# Edits ONLY the target region, using 30-40 steps and negative prompts to resolve hand/tool anatomy
input_image = "assets/realistic_craftsman.png" # @param {type:"string"}
target_object = "the hand saw, wooden block, and hands" # @param {type:"string"}
edit_instruction = "An elderly baker in a clean kitchen cleanly slicing a gourmet chocolate cake with a chef knife, natural hands" # @param {type:"string"}
negative_prompt = "saw teeth, jagged blade, mangled hands, extra fingers, dirty table, wood shavings, cartoon, blurry" # @param {type:"string"}
model_variant = "Base Model (30-50 steps)" # @param ["Base Model (30-50 steps)", "Turbo Model (4 steps)"]
steps = 35 # @param {type:"slider", min:4, max:50, step:1}
cfg_scale = 4.5 # @param {type:"slider", min:1.0, max:10.0, step:0.5}
expand_pixels = 20 # @param {type:"slider", min:0, max:48, step:2}
feather_pixels = 4 # @param {type:"slider", min:0, max:16, step:1}
preview_masks_only = False # @param {type:"boolean"}
output_path = "./output/drive_segmented_cake.png" # @param {type:"string"}

import os
from IPython.display import Image, display

if not os.path.exists("./stable-diffusion.cpp/build/bin/sd-cli"):
    raise FileNotFoundError("Engine binary missing. Please run Cell 2 first.")

model_file = "./models/diffusion_models/Qwen-Image-2.1-Q4_K_M.gguf" if "Base" in model_variant else "./models/diffusion_models/Qwen-Image-2.1-viggle-turbo-Q4_K_M.gguf"

cmd = [
    "python", "edit_segmented.py",
    "--image", input_image,
    "--target", target_object,
    "--instruction", edit_instruction,
    "--output", output_path,
    "--expand-pixels", str(expand_pixels),
    "--feather-pixels", str(feather_pixels),
    "--steps", str(steps),
    "--cfg", str(cfg_scale),
    "--model", model_file,
]

if negative_prompt.strip():
    cmd.extend(["--negative-prompt", negative_prompt.strip()])

if preview_masks_only:
    cmd.append("--preview-masks")

import subprocess
print("Running Segmentation-Aware Editor (weights loaded directly from Google Drive)...")
proc = subprocess.run(cmd)

# Display diagnostic masks and output image inline
base_name = os.path.splitext(os.path.basename(output_path))[0]
diag_dir = os.path.join(os.path.dirname(os.path.abspath(output_path)), f"{base_name}_diagnostics")
three_mask_panel = os.path.join(diag_dir, f"{base_name}_three_mask_panel.png")

if os.path.exists(three_mask_panel):
    print("\n--- Diagnostic Three-Mask Panel (Target, Allowed, Protected) ---")
    display(Image(three_mask_panel, width=800))

if os.path.exists(output_path) and not preview_masks_only:
    print("\n--- Final Pixel-Preserved Output ---")
    display(Image(output_path, width=800))


In [ ]:
# @title 7. Global Image-to-Image Edit (Direct Drive Loading) { run: "auto" }
input_image = "./output/drive_generation.png" # @param {type:"string"}
edit_prompt = "Replace the wooden block with a chocolate cake, and the saw with a chef knife" # @param {type:"string"}
negative_prompt = "saw teeth, mangled fingers, dirty table" # @param {type:"string"}
output_edit = "./output/drive_edit.png" # @param {type:"string"}
steps = 35 # @param {type:"slider", min:1, max:50, step:1}
cfg_scale = 4.5 # @param {type:"slider", min:1.0, max:15.0, step:0.5}
width = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}
height = 1024 # @param [512, 768, 1024, 1280] {type:"raw"}

import os
from IPython.display import Image, display

if not os.path.exists("./stable-diffusion.cpp/build/bin/sd-cli"):
    raise FileNotFoundError("Engine binary missing. Please run Cell 2 first.")

os.environ["STEPS"] = str(steps)
os.environ["CFG"] = str(cfg_scale)
os.environ["WIDTH"] = str(width)
os.environ["HEIGHT"] = str(height)
if negative_prompt.strip():
    os.environ["NEGATIVE_PROMPT"] = negative_prompt.strip()
else:
    os.environ.pop("NEGATIVE_PROMPT", None)

!./edit_image.sh "{input_image}" "{edit_prompt}" "{output_edit}"

if os.path.exists(output_edit):
    display(Image(output_edit))
